In [1]:
from google.colab import drive
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
# 1. Mount Google Drive (Required since this is a new notebook)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 2. Define Paths
BASE_DIR = "/content/drive/MyDrive/Data Science Project/raw_data"
PROJECT_ROOT = os.path.dirname(BASE_DIR)
master_csv_path = os.path.join(PROJECT_ROOT, "nyc_flights_raw_master_2021_2025.csv")

In [4]:
# 3. Load the dataset into Colab RAM
print("--- Loading Master Dataset ---")
print(f"Reading from: {master_csv_path}")
print("Please wait 2-3 minutes while ~35.9 million rows are loaded into memory...")

flights_df = pd.read_csv(master_csv_path)
print(f"✅ Data successfully loaded! Total rows: {len(flights_df):,}")

--- Loading Master Dataset ---
Reading from: /content/drive/MyDrive/Data Science Project/nyc_flights_raw_master_2021_2025.csv
Please wait 2-3 minutes while ~35.9 million rows are loaded into memory...
✅ Data successfully loaded! Total rows: 35,887,876


In [5]:
# ==========================================
# STEP 3: Inspect Column Headings
# ==========================================
print("--- Raw Dataset Headings ---")

# Extract the columns into a list
columns_list = list(flights_df.columns)

# Print them out in a clean, numbered format
for i, col in enumerate(columns_list, 1):
    print(f"{i:02d}. {col}")

print("-" * 30)
print(f"Total Number of Columns: {len(columns_list)}")
print("-" * 30)

--- Raw Dataset Headings ---
01. YEAR
02. MONTH
03. DAY_OF_MONTH
04. DAY_OF_WEEK
05. FL_DATE
06. ORIGIN
07. DEST
08. DEP_TIME
09. DEP_DELAY
10. DEP_DELAY_NEW
11. CANCELLED
12. CANCELLATION_CODE
13. AIR_TIME
14. DISTANCE
15. CARRIER_DELAY
16. WEATHER_DELAY
17. NAS_DELAY
------------------------------
Total Number of Columns: 17
------------------------------


In [6]:
# 1. Create the DATE column using vectorized conversion
flights_df['DATE'] = pd.to_datetime(flights_df[['YEAR', 'MONTH', 'DAY_OF_MONTH']].rename(
    columns={'YEAR': 'year', 'MONTH': 'month', 'DAY_OF_MONTH': 'day'}
))

# 2. Drop only MONTH and DAY_OF_MONTH
flights_df.drop(columns=['MONTH', 'DAY_OF_MONTH'], inplace=True)

# Verification
print("✅ DATE column created.")
print("✅ MONTH and DAY_OF_MONTH dropped.")
print(f"Current Columns: {flights_df.columns.tolist()}")

✅ DATE column created.
✅ MONTH and DAY_OF_MONTH dropped.
Current Columns: ['YEAR', 'DAY_OF_WEEK', 'FL_DATE', 'ORIGIN', 'DEST', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'CANCELLED', 'CANCELLATION_CODE', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'DATE']


In [11]:
import numpy as np
import pandas as pd

# ==========================================
# RECOVERY: CREATE PROCESSED COLUMNS
# ==========================================
print("--- 1. Running Cleaning Logic ---")

# Since MONTH/DAY_OF_MONTH are already gone, we use the 'DATE' column you created
if 'DATE' in flights_df.columns:
    flights_df['FL_DATE_PROCESSED'] = pd.to_datetime(flights_df['DATE'])
else:
    flights_df['FL_DATE_PROCESSED'] = pd.to_datetime(flights_df['FL_DATE'])


# Generate the 4-Category Outcome Label
conditions = [
    (flights_df['CANCELLED'] == 1),
    (flights_df['DEP_DELAY_CLEAN'] >= 60),
    (flights_df['DEP_DELAY_CLEAN'] >= 15),
    (flights_df['DEP_DELAY_CLEAN'] < 15) & (flights_df['CANCELLED'] == 0)
]
choices = ['Cancelled', 'Severely Delayed', 'Delayed', 'On-Time']
flights_df['OUTCOME'] = np.select(conditions, choices, default='Unknown')

print("✅ Processed columns created successfully.\n")


--- 1. Running Cleaning Logic ---
✅ Processed columns created successfully.



In [12]:
# Moving FL_DATE_PROCESSED to the first position
cols = ['FL_DATE_PROCESSED'] + [c for c in flights_df.columns if c != 'FL_DATE_PROCESSED']
flights_df = flights_df[cols]

print("✅ Column 'FL_DATE_PROCESSED' is now at the first position.")
print(f"Top 3 Columns: {flights_df.columns[:3].tolist()}")

✅ Column 'FL_DATE_PROCESSED' is now at the first position.
Top 3 Columns: ['FL_DATE_PROCESSED', 'YEAR', 'DAY_OF_WEEK']


In [13]:
# ==========================================
# STEP 5: COMPREHENSIVE SANITY CHECK
# ==========================================
print("--- 2. Column Integrity Check ---")
# Verifying we have all columns (Originals + the 3 new ones)
print(f"Total Columns: {len(flights_df.columns)}")
print("Columns List:", flights_df.columns.tolist())

print("\n--- 3. Outlier Capping Verification ---")
# Comparing raw vs cleaned to prove original data is still there
stats = flights_df[['DEP_DELAY', 'DEP_DELAY_CLEAN']].describe().round(2)
print(stats)
print(f"\nMax of Raw DEP_DELAY: {flights_df['DEP_DELAY'].max()}")
print(f"Max of Cleaned DEP_DELAY: {flights_df['DEP_DELAY_CLEAN'].max()} (Should be 170.0)")

print("\n--- 4. Outcome Classification Breakdown ---")
print(flights_df['OUTCOME'].value_counts())

print("\n--- 5. Null Value Check (Processed Columns) ---")
print(flights_df[['FL_DATE_PROCESSED', 'DEP_DELAY_CLEAN', 'OUTCOME']].isnull().sum())

print("\n--- 6. Data Preview (First 5 Rows) ---")
display(flights_df.head(15))

--- 2. Column Integrity Check ---
Total Columns: 19
Columns List: ['FL_DATE_PROCESSED', 'YEAR', 'DAY_OF_WEEK', 'FL_DATE', 'ORIGIN', 'DEST', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'CANCELLED', 'CANCELLATION_CODE', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'DATE', 'DEP_DELAY_CLEAN', 'OUTCOME']

--- 3. Outlier Capping Verification ---
         DEP_DELAY  DEP_DELAY_CLEAN
count  35291041.00      35887876.00
mean         12.12             9.65
std          54.75            32.99
min        -115.00          -115.00
25%          -6.00            -6.00
50%          -2.00            -2.00
75%           9.00             8.00
max        7223.00           170.00

Max of Raw DEP_DELAY: 7223.0
Max of Cleaned DEP_DELAY: 170.0 (Should be 170.0)

--- 4. Outcome Classification Breakdown ---
OUTCOME
On-Time             28150790
Delayed              4614462
Severely Delayed     2506595
Cancelled             616029
Name: count, dtype: int64

--- 5. Null Value Check (Processed

,FL_DATE_PROCESSED,YEAR,DAY_OF_WEEK,FL_DATE,ORIGIN,DEST,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CANCELLED,CANCELLATION_CODE,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,DATE,DEP_DELAY_CLEAN,OUTCOME
0,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,606.0,-4.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,2022-12-01,-4.0,On-Time
1,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,1236.0,-6.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,2022-12-01,-6.0,On-Time
2,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,1748.0,2.0,2.0,0.0,NaN,104.0,692.0,NaN,NaN,NaN,2022-12-01,2.0,On-Time
3,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,BNA,820.0,-10.0,0.0,0.0,NaN,115.0,685.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
4,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,552.0,-10.0,0.0,0.0,NaN,79.0,481.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
5,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,1148.0,-6.0,0.0,0.0,NaN,82.0,481.0,NaN,NaN,NaN,2022-12-01,-6.0,On-Time
6,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,1933.0,-10.0,0.0,0.0,NaN,78.0,481.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
7,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ORD,720.0,-4.0,0.0,0.0,NaN,113.0,655.0,NaN,NaN,NaN,2022-12-01,-4.0,On-Time
8,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ORD,1713.0,-11.0,0.0,0.0,NaN,106.0,655.0,NaN,NaN,NaN,2022-12-01,-11.0,On-Time
9,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,PIE,1726.0,175.0,175.0,0.0,NaN,138.0,970.0,170.0,0.0,0.0,2022-12-01,170.0,Severely Delayed


In [10]:
# ==========================================
# FINAL COMPLETENESS CHECK (2021-2025)
# ==========================================
print("--- Generating Year/Month Coverage Grid ---")

# 1. Extract the month from your processed date column
flights_df['TEMP_MONTH'] = flights_df['FL_DATE_PROCESSED'].dt.month

# 2. Create a crosstab to count flights per month per year
coverage_grid = pd.crosstab(
    index=flights_df['YEAR'],
    columns=flights_df['TEMP_MONTH'],
    margins=True,
    margins_name="Total Flights"
)

# 3. Clean up the temporary column
flights_df.drop(columns=['TEMP_MONTH'], inplace=True)

# 4. Display the result
print("\n✅ SUCCESS: Monthly Flight Counts Breakdown:")
print("-" * 75)
display(coverage_grid)
print("-" * 75)
print("\nVerify that every cell from 2021 to 2025 has a large flight count.")

--- Generating Year/Month Coverage Grid ---

✅ SUCCESS: Monthly Flight Counts Breakdown:
---------------------------------------------------------------------------


TEMP_MONTH,1,2,3,4,5,6,7,8,9,10,11,12,Total Flights
YEAR,,,,,,,,,,,,,
2021,379384,350170,467126,473936,520059,573779,615703,611494,567916,595373,576693,580238,6311871
2022,563737,519952,590542,580290,602950,602057,618790,613649,580391,595322,567507,578321,7013508
2023,573877,536229,616234,596676,616630,613577,638995,640236,604715,635538,599814,606218,7278739
2024,582425,552691,628786,619940,649428,651799,676807,660639,621649,656283,614597,631944,7546988
2025,599013,559577,664932,644084,667586,674179,696049,666242,621601,668332,630188,644987,7736770
Total Flights,2698436,2518619,2967620,2914926,3056653,3115391,3246344,3192260,2996272,3150848,2988799,3041708,35887876


---------------------------------------------------------------------------

Verify that every cell from 2021 to 2025 has a large flight count.


In [14]:
display(flights_df.head(15))

,FL_DATE_PROCESSED,YEAR,DAY_OF_WEEK,FL_DATE,ORIGIN,DEST,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CANCELLED,CANCELLATION_CODE,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,DATE,DEP_DELAY_CLEAN,OUTCOME
0,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,606.0,-4.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,2022-12-01,-4.0,On-Time
1,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,1236.0,-6.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,2022-12-01,-6.0,On-Time
2,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ATL,1748.0,2.0,2.0,0.0,NaN,104.0,692.0,NaN,NaN,NaN,2022-12-01,2.0,On-Time
3,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,BNA,820.0,-10.0,0.0,0.0,NaN,115.0,685.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
4,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,552.0,-10.0,0.0,0.0,NaN,79.0,481.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
5,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,1148.0,-6.0,0.0,0.0,NaN,82.0,481.0,NaN,NaN,NaN,2022-12-01,-6.0,On-Time
6,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,CLT,1933.0,-10.0,0.0,0.0,NaN,78.0,481.0,NaN,NaN,NaN,2022-12-01,-10.0,On-Time
7,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ORD,720.0,-4.0,0.0,0.0,NaN,113.0,655.0,NaN,NaN,NaN,2022-12-01,-4.0,On-Time
8,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,ORD,1713.0,-11.0,0.0,0.0,NaN,106.0,655.0,NaN,NaN,NaN,2022-12-01,-11.0,On-Time
9,2022-12-01,2022,4,12/1/2022 12:00:00 AM,ABE,PIE,1726.0,175.0,175.0,0.0,NaN,138.0,970.0,170.0,0.0,0.0,2022-12-01,170.0,Severely Delayed


In [15]:
# ==========================================
# STEP 5.2: FINAL COLUMN BRANDING
# ==========================================
print("--- Finalizing Date Column Branding ---")

# 1. Rename the processed column to 'DATE'
flights_df.rename(columns={'FL_DATE_PROCESSED': 'DATE'}, inplace=True)

# 2. Remove the original raw flight date column
if 'FL_DATE' in flights_df.columns:
    flights_df.drop(columns=['FL_DATE'], inplace=True)

# 3. Move 'DATE' to the first position
cols = ['DATE'] + [c for c in flights_df.columns if c != 'DATE']
flights_df = flights_df[cols]

print("✅ Column branding and reordering complete.")

# ==========================================
# SANITY CHECK & PREVIEW
# ==========================================
print("\n--- Sanity Check ---")
print(f"Total Columns: {len(flights_df.columns)}")
print(f"Primary Date Column: {flights_df.columns[0]}")

# Verification of top categories
print("\nOutcome Frequency Breakdown:")
print(flights_df['OUTCOME'].value_counts())

print("\n--- Data Preview (Top 5 Rows) ---")
display(flights_df.head())

--- Finalizing Date Column Branding ---
✅ Column branding and reordering complete.

--- Sanity Check ---
Total Columns: 18
Primary Date Column: DATE

Outcome Frequency Breakdown:
OUTCOME
On-Time             28150790
Delayed              4614462
Severely Delayed     2506595
Cancelled             616029
Name: count, dtype: int64

--- Data Preview (Top 5 Rows) ---


,DATE,DATE,YEAR,DAY_OF_WEEK,ORIGIN,DEST,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CANCELLED,CANCELLATION_CODE,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,DEP_DELAY_CLEAN,OUTCOME
0,2022-12-01,2022-12-01,2022,4,ABE,ATL,606.0,-4.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,-4.0,On-Time
1,2022-12-01,2022-12-01,2022,4,ABE,ATL,1236.0,-6.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,-6.0,On-Time
2,2022-12-01,2022-12-01,2022,4,ABE,ATL,1748.0,2.0,2.0,0.0,NaN,104.0,692.0,NaN,NaN,NaN,2.0,On-Time
3,2022-12-01,2022-12-01,2022,4,ABE,BNA,820.0,-10.0,0.0,0.0,NaN,115.0,685.0,NaN,NaN,NaN,-10.0,On-Time
4,2022-12-01,2022-12-01,2022,4,ABE,CLT,552.0,-10.0,0.0,0.0,NaN,79.0,481.0,NaN,NaN,NaN,-10.0,On-Time


In [16]:
# 1. Clear the Index name so it doesn't display 'DATE' on the far left
flights_df.index.name = None

# 2. Check for actual duplicate column names (just to be 100% safe)
duplicate_cols = flights_df.columns[flights_df.columns.duplicated()].tolist()

if duplicate_cols:
    print(f"⚠️ Found actual duplicate columns: {duplicate_cols}")
    # This logic removes duplicate columns if they exist, keeping the first occurrence
    flights_df = flights_df.loc[:, ~flights_df.columns.duplicated()]
    print("✅ Duplicate columns removed.")
else:
    print("✅ No duplicate columns found; the 'Double Date' was just the index label.")

# 3. Final Preview
print("\n--- Final Clean Preview ---")
display(flights_df.head())

⚠️ Found actual duplicate columns: ['DATE']
✅ Duplicate columns removed.

--- Final Clean Preview ---


,DATE,YEAR,DAY_OF_WEEK,ORIGIN,DEST,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CANCELLED,CANCELLATION_CODE,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,DEP_DELAY_CLEAN,OUTCOME
0,2022-12-01,2022,4,ABE,ATL,606.0,-4.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,-4.0,On-Time
1,2022-12-01,2022,4,ABE,ATL,1236.0,-6.0,0.0,0.0,NaN,106.0,692.0,NaN,NaN,NaN,-6.0,On-Time
2,2022-12-01,2022,4,ABE,ATL,1748.0,2.0,2.0,0.0,NaN,104.0,692.0,NaN,NaN,NaN,2.0,On-Time
3,2022-12-01,2022,4,ABE,BNA,820.0,-10.0,0.0,0.0,NaN,115.0,685.0,NaN,NaN,NaN,-10.0,On-Time
4,2022-12-01,2022,4,ABE,CLT,552.0,-10.0,0.0,0.0,NaN,79.0,481.0,NaN,NaN,NaN,-10.0,On-Time


In [17]:
# ==========================================
# STEP 6: ISOLATE NYC AIRPORTS (JFK, LGA, EWR)
# ==========================================
print("--- Filtering for NYC Hubs ---")

# 1. Define the target airports
nyc_airports = ['JFK', 'LGA', 'EWR']

# 2. Record starting row count
initial_rows = len(flights_df)

# 3. Filter: Keep rows where NYC is the Origin OR the Destination
# This ensures we capture all traffic flowing in and out of the tri-state area
flights_df = flights_df[
    (flights_df['ORIGIN'].isin(nyc_airports)) |
    (flights_df['DEST'].isin(nyc_airports))
].copy()

# 4. Results calculation
final_rows = len(flights_df)
rows_removed = initial_rows - final_rows

print(f"✅ Filter Applied: {final_rows:,} rows remaining.")
print(f"📉 Removed {rows_removed:,} irrelevant rows (Flights not involving NYC).")

# ==========================================
# BREAKDOWN BY AIRPORT
# ==========================================
print("\nFlight Distribution at NYC Hubs:")
print("-" * 30)
# Count flights where the airport appears as Origin
origin_counts = flights_df['ORIGIN'].value_counts().reindex(nyc_airports).fillna(0).astype(int)
# Count flights where the airport appears as Destination
dest_counts = flights_df['DEST'].value_counts().reindex(nyc_airports).fillna(0).astype(int)

# Create a small summary table
summary_df = pd.DataFrame({
    'Flights as Origin': origin_counts,
    'Flights as Dest': dest_counts,
    'Total Impact': origin_counts + dest_counts
})
print(summary_df)
print("-" * 30)

--- Filtering for NYC Hubs ---
✅ Filter Applied: 4,064,563 rows remaining.
📉 Removed 31,823,313 irrelevant rows (Flights not involving NYC).

Flight Distribution at NYC Hubs:
------------------------------
     Flights as Origin  Flights as Dest  Total Impact
JFK             600729           600633       1201362
LGA             755191           755168       1510359
EWR             676422           676421       1352843
------------------------------
